# Video Game Sales Prediction Pipeline
**Objective:** Develop a regression model to estimate global video game sales based on categorical features (Platform, Genre, Publisher).
**Stack:** pandas, scikit-learn, numpy

In [123]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Data Preprocessing
Loading the dataset and removing features that cause data leakage (regional sales). Dropping identifiers that lack predictive power.

In [124]:
vgsales_df = pd.read_csv('data/vgsales.csv')

# Retain only pre-release features
df_clean = vgsales_df.drop(columns=['Rank', 'Name', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales'])

# Handle missing values to prevent matrix operations failure in scikit-learn
df_clean = df_clean.dropna()

### Feature Engineering
Converting categorical variables into a sparse matrix using One-Hot Encoding.

In [125]:
df_encoded = pd.get_dummies(df_clean, dtype=int)

### Train/Test Validation Split
Isolating the target variable and splitting the dataset (80/20) to validate model generalization.

In [126]:
X = df_encoded.drop(columns='Global_Sales')
y = df_encoded['Global_Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Baseline Model: Linear Regression
Establishing a performance baseline. Applying post-processing (`np.clip`) to prevent negative sales predictions.

In [127]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, a_min=0, a_max=None)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Linear Regression (Baseline) ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}\n")

--- Linear Regression (Baseline) ---
MAE:  0.5454
RMSE: 1.9740
R2:   0.0892



### Advanced Model: Random Forest
Utilizing an ensemble method to capture non-linear relationships and handle potential outliers. Depth is restricted to prevent overfitting.

In [128]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_rf = np.clip(y_pred_rf, a_min=0, a_max=None)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("--- Random Forest (max_depth=15) ---")
print(f"MAE:  {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R2:   {r2_rf:.4f}\n")

--- Random Forest (max_depth=15) ---
MAE:  0.5273
RMSE: 2.0209
R2:   0.0453



### Overfitting Analysis
Testing the Random Forest model without depth constraints to observe variance and overfitting behavior on the test set.

In [129]:
rf_deep = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_deep.fit(X_train, y_train)

y_pred_rf_deep = rf_deep.predict(X_test)
y_pred_rf_deep = np.clip(y_pred_rf_deep, a_min=0, a_max=None)

mae_rf_deep = mean_absolute_error(y_test, y_pred_rf_deep)
rmse_rf_deep = np.sqrt(mean_squared_error(y_test, y_pred_rf_deep))
r2_rf_deep = r2_score(y_test, y_pred_rf_deep)

print("--- Random Forest (Unrestricted Depth) ---")
print(f"MAE:  {mae_rf_deep:.4f}")
print(f"RMSE: {rmse_rf_deep:.4f}")
print(f"R2:   {r2_rf_deep:.4f}")

--- Random Forest (Unrestricted Depth) ---
MAE:  0.5185
RMSE: 2.0280
R2:   0.0386


### Conclusions & Technical Insights

**1. Model Performance Analysis:**
* The Random Forest (max_depth=15) achieved the lowest MAE (~0.52), meaning it predicts "average" games more accurately than the baseline.
* However, Linear Regression achieved the lowest RMSE and highest R-squared. Decision trees inherently struggle to extrapolate beyond the maximum target values seen in the training data. Consequently, the Random Forest heavily underestimated extreme outliers (blockbuster hits like *GTA* or *Wii Sports*), causing the RMSE to spike.
* The unrestricted Random Forest model overfit the training data, memorizing specific developer/platform combinations without capturing true underlying patterns.

**2. Feature Limitations:**
* The current feature set (Platform, Genre, Publisher) has low predictive power. Knowing a game is an "Action game published by Ubisoft on PS4" is insufficient to determine if it will sell 1 million or 15 million copies.

**3. Proposed Next Steps (Future Work):**
* **Target Transformation:** Video game sales follow a heavily right-skewed distribution. Applying a logarithmic transformation (`np.log1p`) to the `Global_Sales` target before training could significantly reduce the impact of extreme outliers on the RMSE.
* **Data Enrichment:** To improve the R-squared metric, the dataset must be enriched with external variables, such as Metacritic review scores, marketing budgets, or binary flags indicating whether a game is part of an established franchise (e.g., *Mario*, *Call of Duty*).
* **Algorithm Upgrade:** Switch to Gradient Boosting frameworks (e.g., LightGBM or XGBoost), which handle sparse matrices and complex non-linear interactions more efficiently than standard Random Forests.